In [1]:
from dbfread import DBF
import pandas as pd
import numpy as np

## Data preparation
### Raw data ingestion

In [2]:
def read_dbf(file_path):
	# read the raw data and store in a dataframe
	dbf = DBF(file_path)
	df = pd.DataFrame(iter(dbf))

	# identify empty strings as missing values
	df.replace("", np.nan, inplace=True)

	# ensure there are now empty rows or columns
	df.dropna(how='all', axis=0, inplace=True)
	df.dropna(how='all', axis=1, inplace=True)

	# remove any duplicates
	df.drop_duplicates(inplace=True)

	# standardize column names
	df.columns = map(lambda x: x.lower(), df.columns)

	return df

#### Expedition data

In [3]:
exped_df = read_dbf('data/raw/exped.DBF')

In [4]:
exped_df.shape

(11578, 66)

In [5]:
exped_df.head()

,expid,peakid,year,season,host,route1,route2,route3,route4,nation,...,accidents,achievment,agency,comrte,stdrte,primrte,primmem,primref,primid,chksum
0,ANN260101,ANN2,1960,1,1,NW Ridge-W Ridge,NaN,NaN,NaN,UK,...,NaN,NaN,NaN,None,None,False,False,None,NaN,2442047
1,ANN269301,ANN2,1969,3,1,NW Ridge-W Ridge,NaN,NaN,NaN,Yugoslavia,...,Draslar frostbitten hands and feet,NaN,NaN,None,None,False,False,None,NaN,2445501
2,ANN273101,ANN2,1973,1,1,W Ridge-N Face,NaN,NaN,NaN,Japan,...,NaN,NaN,NaN,None,None,False,False,None,NaN,2446797
3,ANN278301,ANN2,1978,3,1,N Face-W Ridge,NaN,NaN,NaN,UK,...,NaN,NaN,NaN,None,None,False,False,None,NaN,2448822
4,ANN279301,ANN2,1979,3,1,N Face-W Ridge,NW Ridge of A-IV,NaN,NaN,UK,...,NaN,NaN,NaN,None,None,False,False,None,NaN,2449204


### Data Cleaning

In [6]:
exped_df.groupby('expid').expid.count().sort_values(ascending=False)[:1]

expid
KANG10101    2
Name: expid, dtype: int64

In [7]:
exped_df.loc[exped_df.expid == 'KANG10101']

,expid,peakid,year,season,host,route1,route2,route3,route4,nation,...,accidents,achievment,agency,comrte,stdrte,primrte,primmem,primref,primid,chksum
2860,KANG10101,KANG,1910,1,3,NE Side (recon),NaN,NaN,NaN,UK,...,NaN,NaN,NaN,None,None,False,False,None,NaN,2405
6780,KANG10101,KANG,2010,1,1,SW Face,NaN,NaN,NaN,S Korea,...,NaN,NaN,Windhorse Trekking,False,True,False,False,False,NaN,2457821


> Some _expid_ values are duplicated for expeditions to the same peak occuring one century appart

In [8]:
# add the full year to the expedition id to ensure uniqueness
exped_df.expid = exped_df.expid.str.cat(exped_df.year)
assert exped_df.expid.nunique() == exped_df.shape[0]

In [9]:
# map country index to name as per the documentation
host_map = {
	0: 'Unknown',
	1: 'Nepal',
	2: 'China',
	3: 'India'
}

exped_df.host = exped_df.host.map(host_map)

In [10]:
# map season index to name as per the documentation
season_map = {
	0: 'Unknown',
	1: 'Spring',
	2: 'Summer',
	3: 'Autumn',
	4: 'Winter'
}

exped_df.season = exped_df.season.map(season_map)

In [11]:
# remove expedition with undefined main route
exped_df = exped_df.loc[exped_df.route1.notna()]

In [12]:
# remove expeditions that include non-climbing activities
exped_df = exped_df.loc[~exped_df.traverse & ~exped_df.ski & ~exped_df.parapente]

exped_df.drop(['traverse', 'ski', 'parapente'], axis=1, inplace=True)

In [13]:
# filter based on expedition termination reason:
# 12 - Did not attempt climb
# 13 - Attempt rumored
exped_df = exped_df.loc[
	~exped_df.termreason.isin([12, 13])
]

exped_df.drop('termreason', axis=1, inplace=True)

In [14]:
# remove unused columns
exped_df.drop([
	'route2', 'route3', 'route4', 'success2', 'success3', 'success4', 'ascent2', 'ascent3', 'ascent4', 'claimed',
	'disputed', 'approach', 'smtdate', 'smttime', 'smtdays', 'totdays', 'termdate', 'termnote', 'highpoint',
	'smtmembers', 'mdeaths', 'smthired', 'hdeaths', 'othersmts', 'campsites', 'routememo', 'accidents', 'achievment',
	'primmem', 'primref', 'primid', 'chksum', 'leaders', 'countries', 'ascent1', 'bcdate', 'o2used', 'o2none',
	'o2medical', 'o2unkwn', 'agency', 'o2taken', 'nohired', 'rope'], axis=1, inplace=True)

In [15]:
exped_df.shape

(10888, 18)

> After initial cleaning and filtering, we are left with a dataset of 10,888 expeditions

In [16]:
# exped_df.leaders = exped_df.leaders.str.split(',')
# exped_df['leader_count'] = exped_df.leaders.apply(lambda x: len(x))
# exped_df.drop('leaders', axis=1, inplace=True)

In [17]:
# create flag variables to indicate whether the expedition has a sponsor
exped_df['sponsored'] = exped_df.sponsor.notna()
exped_df.drop('sponsor', axis=1, inplace=True)

In [18]:
# concatenate peak and route
exped_df['ascent_route'] = exped_df.peakid.str.cat(exped_df.route1, sep='-').str.replace(" ", "_")
exped_df.drop(['peakid', 'route1'], axis=1, inplace=True)

In [19]:
# find most commonly attempted ascents
route_counts = pd.DataFrame(exped_df.ascent_route.value_counts()).reset_index()
common_routes = route_counts.loc[route_counts['count'] >= 10, 'ascent_route']

In [20]:
# keep only expeditions on common routes
exped_df = exped_df.loc[exped_df.ascent_route.isin(common_routes)]

In [21]:
exped_df.shape

(7900, 17)

In [22]:
exped_df.head()

,expid,year,season,host,nation,success1,camps,totmembers,tothired,o2climb,o2descent,o2sleep,comrte,stdrte,primrte,sponsored,ascent_route
21,ANN4502011950,1950,Summer,Nepal,UK,False,4,6,5,False,False,False,None,None,False,True,ANN4-NW_Ridge
25,ANN4551011955,1955,Spring,Nepal,W Germany,True,4,4,2,False,False,False,None,None,False,True,ANN4-NW_Ridge
26,ANN4571011957,1957,Spring,Nepal,UK,True,4,2,4,False,False,False,None,None,False,False,ANN4-NW_Ridge
27,ANN4601011960,1960,Spring,Nepal,UK,True,5,4,3,False,False,False,None,None,False,False,ANN4-NW_Ridge
28,ANN4693011969,1969,Autumn,Nepal,Czechoslovakia,True,4,9,1,False,False,False,None,None,False,True,ANN4-NW_Ridge


In [23]:
exped_df.success1.value_counts()

success1
True     4893
False    3007
Name: count, dtype: int64

#### Climber data

In [24]:
climber_df = read_dbf('data/raw/members.DBF')

In [25]:
climber_df.head()

,expid,membid,peakid,myear,mseason,fname,lname,sex,age,yob,...,membermemo,necrology,msmtbid,msmtterm,hcn,mchksum,msmtnote1,msmtnote2,msmtnote3,deathrte
0,AMAD78301,01,AMAD,1978,3,Jean Robert,Clemenson,M,0,1938,...,None,None,1,4,0,2426937,NaN,NaN,NaN,NaN
1,AMAD78301,02,AMAD,1978,3,Bernard,Dufour,M,0,1936,...,None,None,1,4,0,2426501,NaN,NaN,NaN,NaN
2,AMAD78301,03,AMAD,1978,3,Philippe,Gerard,M,0,1950,...,None,None,1,4,0,2431569,NaN,NaN,NaN,NaN
3,AMAD78301,04,AMAD,1978,3,Eric,Lasserre,M,0,1937,...,None,None,1,4,0,2426809,NaN,NaN,NaN,NaN
4,AMAD78301,05,AMAD,1978,3,Guy,Peters,M,0,1944,...,None,None,1,4,0,2429215,NaN,NaN,NaN,NaN


In [26]:
climber_df = climber_df[[
	'expid', 'myear', 'mseason', 'fname', 'lname', 'yob', 'status', 'leader', 'support', 'disabled', 'hired', 'sherpa',
	'msuccess']].dropna(how='any', subset=['expid', 'myear', 'fname', 'lname', 'yob'])

In [27]:
climber_df.dtypes

expid       object
myear       object
mseason      int64
fname       object
lname       object
yob         object
status      object
leader        bool
support       bool
disabled      bool
hired         bool
sherpa        bool
msuccess      bool
dtype: object

In [28]:
climber_df.expid = climber_df.expid.str.cat(climber_df.myear)
climber_df['age'] = climber_df.myear.astype(int) - climber_df.yob.astype(int)

In [29]:
climber_df.sort_values(['fname', 'lname', 'yob', 'myear', 'mseason'], inplace=True)

In [30]:
climber_df['experience'] = climber_df.groupby(['fname', 'lname', 'yob'], as_index=False).expid.cumcount()
climber_df['leadership_experience'] = climber_df.groupby(['fname', 'lname', 'yob'], as_index=False).leader.cumsum()
climber_df['successes'] = climber_df.groupby(['fname', 'lname', 'yob'], as_index=False).msuccess.cumsum()

In [31]:
team_df = climber_df.groupby('expid', as_index=False).agg({
	'experience': 'sum',
	'leadership_experience': 'sum',
	'successes': 'sum',
	'age': 'mean',
	'leader': 'sum',
	'support': 'sum',
	'disabled': 'sum',
	'hired': 'sum',
	'sherpa': 'sum'
})

In [32]:
team_df.head()

,expid,experience,leadership_experience,successes,age,leader,support,disabled,hired,sherpa
0,ACHN153012015,0,1,5,22.600000,1,0,0,0,0
1,ACHN153022015,68,44,33,49.181818,1,0,0,2,0
2,ACHN183012018,33,5,16,44.555556,2,0,0,1,0
3,AMAD001012000,0,1,5,31.200000,1,0,0,0,0
4,AMAD001022000,1,1,5,39.000000,1,0,0,0,0


In [33]:
team_df.rename({
	'experience': 'total_experience',
	'leadership_experience': 'total_leadership_experience',
	'successes': 'total_successes',
	'leader': 'leaders',
	'sherpa': 'sherpas',
	'age': 'avg_climber_age'
}, axis=1, inplace=True)

In [34]:
exped_cols = set(exped_df.columns)
team_cols = set(team_df.columns)

In [35]:
team_df.shape

(11434, 10)

In [36]:
df = exped_df.merge(team_df, how='inner')

In [37]:
df.shape

(7871, 26)

In [38]:
df.head()

,expid,year,season,host,nation,success1,camps,totmembers,tothired,o2climb,...,ascent_route,total_experience,total_leadership_experience,total_successes,avg_climber_age,leaders,support,disabled,hired,sherpas
0,ANN4502011950,1950,Summer,Nepal,UK,False,4,6,5,False,...,ANN4-NW_Ridge,12,6,2,34.727273,1,0,0,5,5
1,ANN4551011955,1955,Spring,Nepal,W Germany,True,4,4,2,False,...,ANN4-NW_Ridge,0,1,3,26.750000,1,0,0,0,0
2,ANN4571011957,1957,Spring,Nepal,UK,True,4,2,4,False,...,ANN4-NW_Ridge,21,6,10,34.500000,1,0,0,0,0
3,ANN4601011960,1960,Spring,Nepal,UK,True,5,4,3,False,...,ANN4-NW_Ridge,13,9,5,32.000000,1,0,0,1,1
4,ANN4693011969,1969,Autumn,Nepal,Czechoslovakia,True,4,9,1,False,...,ANN4-NW_Ridge,0,0,0,33.000000,0,0,0,0,0


In [39]:
assert df.expid.nunique() == df.shape[0]

In [40]:
df.support.describe()

count    7871.000000
mean        0.009529
std         0.148814
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         6.000000
Name: support, dtype: float64